<a href="https://colab.research.google.com/github/joseguzman1337/aws-financial-ai-agent/blob/main/invocation_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AWS Financial AI Agent - Invocation Demo

This notebook demonstrates end-to-end user authentication, live agent invocation via AWS Agentcore, and observability trace extraction.

### 0. Install Dependencies & Setup
Ensure the environment has the required libraries and initialize variables.

In [16]:
!pip install boto3 requests

import boto3
import requests
import json

# Initialize variables to prevent NameError in later cells
access_token = None

### 1. Authenticate with Amazon Cognito
Authentication is performed against the live Cognito User Pool provisioned via Terraform.

In [17]:
REGION = "us-east-1"
CLIENT_ID = "2r1ik1k110jbu6nfmuoegk5lns"
USERNAME = "analyst_user"
PASSWORD = "SecurePassword123!"
USER_POOL_ID = "us-east-1_example" # Note: Ensure this matches your Terraform output

if CLIENT_ID == "<YOUR_COGNITO_CLIENT_ID>":
    print("❌ ERROR: Please replace <YOUR_COGNITO_CLIENT_ID> with your actual Client ID.")
else:
    client = boto3.client('cognito-idp', region_name=REGION)
    try:
        # Attempt to sign in
        auth_response = client.initiate_auth(
            ClientId=CLIENT_ID,
            AuthFlow='USER_PASSWORD_AUTH',
            AuthParameters={'USERNAME': USERNAME, 'PASSWORD': PASSWORD}
        )
        access_token = auth_response['AuthenticationResult']['AccessToken']
        print(f"✅ Cognito Authentication Successful. Token acquired: {access_token[:30]}...")
    except client.exceptions.UserNotFoundException:
        print(f"ℹ️ User {USERNAME} not found. In a real scenario, you would create the user in the AWS Console or via AdminCreateUser.")
        # Mocking a token for demo purposes if user creation isn't automated here
        access_token = "mock_token_for_demo_purposes"
        print("⚠️ Using a placeholder token for flow demonstration.")
    except Exception as e:
        print(f"❌ Authentication Failed: {str(e)}")

ℹ️ User analyst_user not found. In a real scenario, you would create the user in the AWS Console or via AdminCreateUser.
⚠️ Using a placeholder token for flow demonstration.


### 2. Invoke the Agentcore Streaming Endpoint
This cell calls the live AWS Agentcore runtime. Responses are streamed in real-time.

In [18]:
AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:162187491349:runtime/Financial_Analyst_Agent-s5aas5HZhy"
AGENTCORE_URL = f"https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/{AGENT_ARN}/invocations"

def query_financial_agent(prompt: str):
    if access_token is None:
        print("❌ ERROR: Access token is missing. Please run Cell 1 (Cognito Authentication) successfully first.")
        return

    print(f"\n--- Query: {prompt} ---")

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": "demo-session-12345678901234567890123"
    }

    response = requests.post(
        AGENTCORE_URL,
        headers=headers,
        json={"prompt": prompt},
        stream=True
    )

    if response.status_code != 200:
        print(f"❌ Error {response.status_code}: {response.text}")
        return

    # Process the Server-Sent Events (SSE) stream
    for line in response.iter_lines():
        if line:
            decoded_line = line.decode('utf-8')
            if decoded_line.startswith("data: "):
                data = json.loads(decoded_line[6:])
                print(data.get("event", ""), end="", flush=True)

### 3. Execute Required Financial Queries
Proving the agent's ability to handle real-time data and RAG retrieval.

In [19]:
queries = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
    "Compare Amazon's recent stock performance to what analysts predicted in their reports.",
    "I’m researching AMZN give me the current price and any relevant information about their AI business.",
    "What is the total amount of office space Amazon owned in North America in 2024?"
]

for q in queries:
    query_financial_agent(q)


--- Query: What is the stock price for Amazon right now? ---
❌ Error 404: <UnknownOperationException/>


--- Query: What were the stock prices for Amazon in Q4 last year? ---
❌ Error 404: <UnknownOperationException/>


--- Query: Compare Amazon's recent stock performance to what analysts predicted in their reports. ---
❌ Error 404: <UnknownOperationException/>


--- Query: I’m researching AMZN give me the current price and any relevant information about their AI business. ---
❌ Error 404: <UnknownOperationException/>


--- Query: What is the total amount of office space Amazon owned in North America in 2024? ---
❌ Error 404: <UnknownOperationException/>



### 4. Langfuse Observability Trace Audit
Verify that the agent's reasoning steps were recorded.

In [20]:
# Verifying traces using Langfuse API
# Note: You must provide valid Public and Secret keys for this to work
public_key = 'pk-lf-...'
secret_key = 'sk-lf-...'

langfuse_auth = (public_key, secret_key)

try:
    trace_response = requests.get(
        "https://cloud.langfuse.com/api/public/traces?sessionId=demo-session-12345678901234567890123",
        auth=langfuse_auth
    )

    print("\n--- Langfuse Trace Data ---")
    if trace_response.status_code == 200:
        print(json.dumps(trace_response.json(), indent=2))
    elif trace_response.status_code == 401:
        print("❌ Invalid Langfuse Credentials. Please update pk-lf and sk-lf variables with your project keys.")
    else:
        print(f"❌ Could not fetch traces: {trace_response.status_code}")
except Exception as e:
    print(f"❌ Error connecting to Langfuse: {e}")


--- Langfuse Trace Data ---
❌ Invalid Langfuse Credentials. Please update pk-lf and sk-lf variables with your project keys.
